# Ichimoku Cloud — trade the bull regime (+ fakeout stop)
The **Kumo** (cloud) between Senkou A & B is a wide support/resistance zone (both spans correctly shifted forward, no look-ahead). **BUY** when price closes **above** the cloud (bull regime), **SELL** when it closes **below** the cloud. A **fakeout stop** trims failed breakouts.

Daily + weekly, 10y, $1,000/trade, keeps only **8–15 trade** configs, ranked by **Score = P&L × Win Rate**.

_Note: trend-following, so a ~50% win rate with a high profit factor is normal — the winners are much larger than the losers._

```bash
pip install yfinance pandas numpy matplotlib jupyter
```

In [ ]:
# ================= CONFIG — edit me =================
TICKER     = "AAPL"
YEARS      = 10
START_DATE = None        # e.g. "2018-01-01"  (overrides YEARS if set)
END_DATE   = None        # e.g. "2024-12-31"
CAPITAL    = 1000.0      # $ per trade
MIN_TRADES = 8           # only keep configs that trade at least this many times
MAX_TRADES = 15          # ...and at most this many (your 8-15 requirement)
RANK_BY    = "Score"     # "Score" (PnL x win-rate) | "Total P&L $" | "Win Rate %" | "Profit Factor"

# parameter grids to sweep (Tenkan, Kijun, fakeout Stop)
W_TK=[6,9,12]; W_KJ=[18,22,26,30]; W_STOP=[0.10,0.15,0.25]
D_TK=[9,12,20,26]; D_KJ=[40,52,80,120]; D_STOP=[0.10,0.15,0.25]


In [ ]:
%matplotlib inline
import yfinance as yf
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 30, "display.width", 200)


In [ ]:
def get_data(ticker, interval, years=YEARS, start=START_DATE, end=END_DATE):
    if start:
        df = yf.download(ticker, start=start, end=end, interval=interval, auto_adjust=True, progress=False)
    else:
        df = yf.download(ticker, period=f"{years}y", interval=interval, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[["Open","High","Low","Close","Volume"]].dropna().reset_index()
    df.rename(columns={df.columns[0]: "Date"}, inplace=True)
    return df

daily  = get_data(TICKER, "1d")
weekly = get_data(TICKER, "1wk")
print(f"{TICKER}: {len(daily)} daily, {len(weekly)} weekly candles "
      f"({daily.Date.min().date()} -> {daily.Date.max().date()})")


In [ ]:
CAPITAL = float(CAPITAL)

def compute_stats(t, capital=CAPITAL):
    z = {"Total Trades":0,"Win Rate %":0,"Total P&L $":0,"Avg Win $":0,"Avg Loss $":0,
         "Profit Factor":0,"Expectancy $":0,"Avg Bars Held":0,"Max Drawdown $":0,"Return/Trade %":0}
    if t.empty: return z
    w = t[t.pnl>0]; l = t[t.pnl<=0]
    gp = w.pnl.sum(); gl = abs(l.pnl.sum())
    eq = t.pnl.cumsum(); dd = (eq-eq.cummax()).min()
    return {"Total Trades":len(t),
            "Win Rate %":round(len(w)/len(t)*100,1),
            "Total P&L $":round(t.pnl.sum(),2),
            "Avg Win $":round(w.pnl.mean(),2) if len(w) else 0.0,
            "Avg Loss $":round(l.pnl.mean(),2) if len(l) else 0.0,
            "Profit Factor":round(gp/gl,2) if gl>0 else float("inf"),
            "Expectancy $":round(t.pnl.mean(),2),
            "Avg Bars Held":round(t.bars.mean(),1),
            "Max Drawdown $":round(dd,2),
            "Return/Trade %":round(t.ret.mean()*100,2)}

def filter_window(res, lo=MIN_TRADES, hi=MAX_TRADES, rank_by=RANK_BY):
    # keep only configs that trade lo..hi times; if none, fall back to nearest to the midpoint
    w = res[(res["Total Trades"]>=lo) & (res["Total Trades"]<=hi)].copy()
    if w.empty:
        mid = (lo+hi)/2
        res = res.assign(_d=(res["Total Trades"]-mid).abs())
        w = res.sort_values("_d").head(8).drop(columns="_d").copy()
    w["Score"] = w["Total P&L $"] * (w["Win Rate %"]/100.0)   # blended PnL x win-rate
    asc = rank_by in ("Max Drawdown $",)
    return w.sort_values(rank_by, ascending=asc).reset_index(drop=True)

def plot_trades(d, trades, lines, fill, title, stats):
    fig,(ax1,ax2) = plt.subplots(2,1,figsize=(16,10),sharex=True,gridspec_kw={"height_ratios":[3,1.3]})
    ax1.plot(d.Date, d.Close, color="#222", lw=1.1, label="Close")
    for col,lbl,c in lines:
        ax1.plot(d.Date, d[col], color=c, lw=1.2, label=lbl, alpha=0.9)
    if fill:
        b,tp,lbl = fill; ax1.fill_between(d.Date, d[b], d[tp], color="#2ca02c", alpha=0.15, label=lbl)
    if not trades.empty:
        ax1.scatter(trades.entry_date, trades.entry_price, marker="^", s=120, color="green", edgecolor="k", zorder=5, label="BUY")
        ax1.scatter(trades.exit_date,  trades.exit_price,  marker="v", s=120, color="red",   edgecolor="k", zorder=5, label="SELL")
        for _,r in trades.iterrows():
            ax1.plot([r.entry_date,r.exit_date],[r.entry_price,r.exit_price], color="green" if r.pnl>0 else "red", lw=1.0, alpha=0.5)
    ax1.set_title(title, fontsize=14, weight="bold"); ax1.set_ylabel("Price ($)")
    ax1.legend(loc="upper left", ncol=2, fontsize=9); ax1.grid(alpha=0.25)
    if not trades.empty:
        cum = trades.pnl.cumsum().values
        ax2.plot(trades.exit_date, cum, color="#1f77b4", marker="o", ms=3, lw=1.3); ax2.axhline(0, color="grey", lw=0.8)
        ax2.fill_between(trades.exit_date, cum, 0, where=cum>=0, color="green", alpha=0.2)
        ax2.fill_between(trades.exit_date, cum, 0, where=cum<0,  color="red",   alpha=0.2)
    ax2.set_ylabel("Cumulative P&L ($)"); ax2.set_xlabel("Date"); ax2.grid(alpha=0.25)
    ax2.xaxis.set_major_locator(mdates.YearLocator()); ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    items = list(stats.items()); half = (len(items)+1)//2
    l1 = "  |  ".join(f"{k}: {v}" for k,v in items[:half]); l2 = "  |  ".join(f"{k}: {v}" for k,v in items[half:])
    fig.text(0.5, 0.005, l1+"\n"+l2, ha="center", va="bottom", fontsize=10, family="monospace",
             bbox=dict(boxstyle="round", fc="#f5f5f5", ec="#999"))
    fig.tight_layout(rect=[0,0.06,1,1]); plt.show()

def trade_log(trades):
    cols = [c for c in ["entry_date","entry_price","exit_date","exit_price","ret","pnl","bars","reason","status"] if c in trades.columns]
    return trades[cols]

def run_bt(d, stop_pct, capital=CAPITAL):
    # d must have boolean columns "entry" and "exit". Fills on next bar open. Fakeout stop = stop_pct.
    trades = []; pos = None
    for i in range(len(d)-1):
        r = d.iloc[i]; nxt = d.iloc[i+1]
        if pos is None:
            if bool(r["entry"]):
                pos = {"entry_date":nxt.Date, "entry_price":nxt.Open, "entry_i":i+1}
        else:
            stop_hit = r.Close <= pos["entry_price"]*(1-stop_pct)
            if bool(r["exit"]) or stop_hit:
                ep = nxt.Open; ret = (ep-pos["entry_price"])/pos["entry_price"]
                trades.append({**pos, "exit_date":nxt.Date, "exit_price":ep, "ret":ret,
                               "pnl":ret*capital, "bars":(i+1)-pos["entry_i"],
                               "reason":"fakeout-stop" if stop_hit else "exit-signal"})
                pos = None
    if pos is not None:
        last = d.iloc[-1]; ret = (last.Close-pos["entry_price"])/pos["entry_price"]
        trades.append({**pos, "exit_date":last.Date, "exit_price":last.Close, "ret":ret,
                       "pnl":ret*capital, "bars":(len(d)-1)-pos["entry_i"], "reason":"open"})
    return pd.DataFrame(trades)

def simple_live(d, stop_pct):
    r = d.iloc[-1]
    # crude in-position check: replay
    t = run_bt(d, stop_pct)
    in_pos = (not t.empty) and (t.iloc[-1]["reason"] == "open")
    if in_pos:
        ep = t.iloc[-1]["entry_price"]
        if r.Close <= ep*(1-stop_pct): rec = "SELL (fakeout stop hit)"
        elif bool(r["exit"]):          rec = "SELL (exit signal)"
        else:                           rec = "HOLD (in position)"
        state = "IN position"
    else:
        rec = "BUY (entry signal - act at next open)" if bool(r["entry"]) else "HOLD (flat - waiting for entry)"
        state = "FLAT"
    return {"date":str(pd.Timestamp(r.Date).date()), "close":round(float(r.Close),2),
            "state":state, "recommendation":rec}

def _cloud(d, tenkan, kijun, senkou_b_len, disp):
    hi, lo = d["High"], d["Low"]
    conv = (hi.rolling(tenkan).max()+lo.rolling(tenkan).min())/2
    base = (hi.rolling(kijun).max()+lo.rolling(kijun).min())/2
    d["span_a"] = ((conv+base)/2).shift(disp)
    d["span_b"] = ((hi.rolling(senkou_b_len).max()+lo.rolling(senkou_b_len).min())/2).shift(disp)
    d["cloud_top"] = d[["span_a","span_b"]].max(axis=1)
    d["cloud_bot"] = d[["span_a","span_b"]].min(axis=1)
    return d

def ichi_prep(df, tenkan, kijun):
    d = _cloud(df.copy(), tenkan, kijun, 2*kijun, kijun).dropna().reset_index(drop=True)
    d["band_top"] = d["cloud_top"]; d["band_bot"] = d["cloud_bot"]
    d["entry"] = d.Close > d["cloud_top"]   # break above the cloud (bull regime)
    d["exit"]  = d.Close < d["cloud_bot"]   # close below the cloud (regime lost)
    return d

def ichi_sweep(df, tk_grid, kj_grid, stop_grid):
    rows = []
    for tk in tk_grid:
        for kj in kj_grid:
            d = ichi_prep(df, tk, kj)
            for st in stop_grid:
                s = compute_stats(run_bt(d, st))
                if s["Total Trades"]==0: continue
                rows.append({"Tenkan":tk,"Kijun":kj,"Stop":st, **s})
    return pd.DataFrame(rows)

def run_ichi(df, tf, tk_grid, kj_grid, stop_grid):
    sw = ichi_sweep(df, tk_grid, kj_grid, stop_grid)
    inwin = int(((sw["Total Trades"]>=MIN_TRADES)&(sw["Total Trades"]<=MAX_TRADES)).sum()); w = filter_window(sw)
    tag = f"{inwin} with {MIN_TRADES}-{MAX_TRADES} trades" + ("" if inwin>0 else f" (none in range -> {len(w)} nearest)")
    print(f"\n=== ICHIMOKU {tf} : {len(sw)} combos, {tag}. Top 5 by {RANK_BY}: ===")
    display(w.head(5)[["Tenkan","Kijun","Stop","Total Trades","Win Rate %","Total P&L $","Profit Factor","Score"]])
    b = w.iloc[0]; d = ichi_prep(df, int(b.Tenkan), int(b.Kijun)); t = run_bt(d, float(b.Stop))
    title = f"{TICKER} - Ichimoku {tf} BEST  (Tenkan {int(b.Tenkan)} / Kijun {int(b.Kijun)}, stop={b.Stop})"
    plot_trades(d, t, [("span_a","Senkou A","#2ca02c"),("span_b","Senkou B","#d62728")],
                ("cloud_bot","cloud_top","Kumo (Cloud)"), title, compute_stats(t))
    display(trade_log(t))
    return {"tenkan":int(b.Tenkan),"kijun":int(b.Kijun),"stop":float(b.Stop)}


## Weekly — sweep, top 5 (8–15 trades), chart the best

In [ ]:
best_w = run_ichi(weekly, 'Weekly', W_TK, W_KJ, W_STOP)

## Daily — sweep, top 5 (8–15 trades), chart the best

In [ ]:
best_d = run_ichi(daily, 'Daily', D_TK, D_KJ, D_STOP)

## Daily signal — run me every day → BUY / SELL / HOLD

In [ ]:
c = best_w
wk_live = get_data(TICKER,'1wk')
d = ichi_prep(wk_live, c['tenkan'], c['kijun'])
sig = simple_live(d, c['stop'])
print(f"===== {TICKER} ICHIMOKU SIGNAL - {sig['date']} =====")
print(f"Close {sig['close']} | {sig['state']}")
print('>>>', sig['recommendation'])